# Postprocessing GOEA results

This notebook contains code to postprocess the GOEA result, in order to investigate functional relevance of gene losses.

In [1]:
from collections import Counter
from dendropy import Tree
import gzip 
import numpy as np
import os
import pandas as pd

from eagle.results import parse_to_concat

In [39]:
# !!! replace this with the tree_X_colouring.tsv file !!!

def get_branchgroup_counts(folder_name):
    df = get_node_labelling(folder_name)
    
    tree_name = '_'.join(folder_name.split('_')[2:])
    t = Tree.get_from_path(f'../labelled_trees/{tree_name}.nwk', schema='newick')
    
    df = pd.read_csv(f'../labelled_trees/{tree_name}_colouring.tsv', sep='\t')[['id', 'classification']]
    df = df.rename(columns={'classification': 'branch_grouping', 'id': 'child_name'})
    internal_nodes = set(map(lambda n: n.label, t.internal_nodes()))
    internal_nodes.remove(t.seed_node.label)
    df = df[df.child_name.isin(internal_nodes)]
    return Counter(df.branch_grouping)
    
def get_node_labelling(folder_name):
    # want to label branches as on the way TO the node of interest (i.e., branch leading to SLL belongs to SLL.)
    tree_name = '_'.join(folder_name.split('_')[2:])
    
    df = pd.read_csv(f'../labelled_trees/{tree_name}_colouring.tsv', sep='\t')[['id', 'classification']]
    df = df.rename(columns={'classification': 'branch_grouping', 'id': 'child_name'})
    return df


def concat_results(folder_name):
    path = os.path.join('/work/FAC/FBM/DBC/cdessim2/default/awarwick/tomato_project/', folder_name)

    clade_labelling_df = get_node_labelling(folder_name)
    
    goea_path = os.path.join(path, 'eagle_results')
    fns = list(map(lambda x: os.path.join(goea_path, x),
                   filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                          os.listdir(goea_path))))
    df = parse_to_concat(*fns)
    
    # label the clade for the branch
    assert len(set(df.child_name) - set(clade_labelling_df.child_name)) == 0
    df = pd.merge(df, clade_labelling_df, on='child_name', how='left')

    branchgroup_counts = get_branchgroup_counts(folder_name)
    df['p_fdr_bh_branchgroup_adjusted'] = df.apply(lambda x: x.p_fdr_bh * branchgroup_counts[x.branch_grouping], axis=1)
    df['p_bonferroni_branchgroup_adjusted'] = df.apply(lambda x: x.p_bonferroni * branchgroup_counts[x.branch_grouping], axis=1)
    
    with gzip.open(os.path.join(path, 'eagle_results_with_ancestral_genes.tsv.gz'), 'wt') as fp:
        df.to_csv(fp, sep='\t', index=False)

    with gzip.open(os.path.join(path, 'eagle_results.tsv.gz'), 'wt') as fp:
        header = [x for x in list(df) if x != 'study_entries_with_go_term']
        df[header].to_csv(fp, sep='\t', index=False)

def concat_results_path(folder_name):
    path = os.path.join('/work/FAC/FBM/DBC/cdessim2/default/awarwick/tomato_project/', folder_name)

    goea_path = os.path.join(path, 'eagle_results_branch_paths')
    fns = list(map(lambda x: os.path.join(goea_path, x),
                   filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                          os.listdir(goea_path))))
    df = parse_to_concat(*fns)
    
    with gzip.open(os.path.join(path, 'eagle_results_branch_paths_with_ancestral_genes.tsv.gz'), 'wt') as fp:
        df.to_csv(fp, sep='\t', index=False)

    with gzip.open(os.path.join(path, 'eagle_results_branch_paths.tsv.gz'), 'wt') as fp:
        header = [x for x in list(df) if x != 'study_entries_with_go_term']
        df[header].to_csv(fp, sep='\t', index=False)

def concat_results_keynode(folder_name):
    path = os.path.join('/work/FAC/FBM/DBC/cdessim2/default/awarwick/tomato_project/', folder_name)
    
    # link based on the key node labelling, instead of using the single-branch labels
    keynode_df = pd.read_csv(os.path.join(path, 'results_loss_analysis/onestep_higher_paths.tsv'), sep='\t')
    keynode_df = keynode_df[['key_node_label', 'child_classification', 'parent_genome_name', 'child_genome_name']]
    keynode_df = keynode_df.rename(columns={'parent_genome_name': 'parent_name', 'child_genome_name': 'child_name'})
    # count number of paths for each key node
    keynode_counts = keynode_df['key_node_label'].value_counts().to_dict()
    
    goea_path = os.path.join(path, 'eagle_single')
    fns = list(map(lambda x: os.path.join(goea_path, x),
                   filter(lambda x: not x.endswith('.extant_genelist.tsv.gz'),
                          os.listdir(goea_path))))
    df = parse_to_concat(*fns)
    
    df = pd.merge(df, keynode_df, on=['parent_name', 'child_name'], how='left')
    
    df['p_fdr_bh_keynodepath_adjusted'] = df.apply(lambda x: min(1, x.p_fdr_bh * keynode_counts[x.key_node_label]), axis=1)
    df['p_bonferroni_keynodepath_adjusted'] = df.apply(lambda x: min(1, x.p_bonferroni * keynode_counts[x.key_node_label]), axis=1)
    
    with gzip.open(os.path.join(path, 'eagle_keynode_results_with_ancestral_genes.tsv.gz'), 'wt') as fp:
        df.to_csv(fp, sep='\t', index=False)

    with gzip.open(os.path.join(path, 'eagle_keynode_results.tsv.gz'), 'wt') as fp:
        header = [x for x in list(df) if x != 'study_entries_with_go_term']
        df[header].to_csv(fp, sep='\t', index=False)

In [38]:
concat_results_keynode('fastoma_round2_tree_a')
concat_results_keynode('fastoma_round2_tree_c')

  0%|          | 0/344 [00:00<?, ?it/s]

In [3]:
concat_results('fastoma_round2_tree_a')
#concat_results('fastoma_round2_tree_b')
concat_results('fastoma_round2_tree_c')
#concat_results('fastoma_round2_tree_d_unfiltered')
#concat_results('fastoma_round2_tree_e_pfam_filtered')
#concat_results('fastoma_round2_tree_f_gopfam_filtered')

  0%|          | 0/248 [00:00<?, ?it/s]

  0%|          | 0/248 [00:00<?, ?it/s]

In [ ]:
concat_results_path('fastoma_round2_tree_a')
concat_results_path('fastoma_round2_tree_b')
concat_results_path('fastoma_round2_tree_c')
concat_results_path('fastoma_round2_tree_d_unfiltered')
concat_results_path('fastoma_round2_tree_e_pfam_filtered')
concat_results_path('fastoma_round2_tree_f_gopfam_filtered')

- number of genes in the root hog

- number of members per root hog

- number of copies at a particular level per hog

In [ ]:
- are large families more likely to experience at least 1 loss?
- do large families lose more copies than expected?


In [ ]:
per root hog
- compute copy number at ancestral / child



for each branch, for each hog:
    - number of members of parent;
    - number of members of child;
    - number of losses on branch;
    - number of extant members in root hog
    - loss_rate = # losses / # members at parent